# GTSRB — chạy trên Google Colab

**Cách dùng: bật GPU rồi bấm `Runtime → Run all`. Hết.**

Notebook tự làm mọi thứ: tải code, tải dữ liệu, chạy 30 run ablation, chạy thêm seed,
đánh giá, rồi tải kết quả về máy bạn.

| | |
|---|---|
| Thời gian | **~2–2,5 giờ** trên T4 (trên MacBook là ~6 giờ) |
| Bạn phải bấm | **2 lần**: bật GPU, và cho phép truy cập Drive |
| Mất session giữa chừng | không sao — chạy lại Run all, nó bỏ qua phần đã xong |

---

## ⚠️ Làm ĐÚNG 2 việc này trước khi Run all

**1. Bật GPU**
`Runtime` → `Change runtime type` → `Hardware accelerator: T4 GPU` → `Save`

**2. Bấm Run all**
`Runtime` → `Run all`  (hoặc `Ctrl+F9`)

Khi chạy tới ô Drive, sẽ hiện popup xin quyền → bấm **Connect to Google Drive** →
chọn tài khoản → kéo xuống → **Allow**. Đó là lần bấm duy nhất.

> Nếu quên bật GPU, ô đầu tiên sẽ dừng lại và báo rõ — không chạy nhầm hàng chục tiếng trên CPU.

## 1 — Kiểm GPU

Ô này **cố ý dừng cả notebook** nếu chưa bật GPU, để bạn không chạy nhầm trên CPU.

In [ ]:
import torch, subprocess

print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit(
        "\n>>> CHƯA BẬT GPU — notebook dừng ở đây. <<<\n\n"
        "Làm: Runtime -> Change runtime type -> T4 GPU -> Save\n"
        "Rồi bấm lại Runtime -> Run all.\n\n"
        "Chạy ablation trên CPU mất hàng chục tiếng thay vì 2 tiếng."
    )

gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU  : {gpu}  ({vram:.0f} GB VRAM)")
print("\nOK, chạy tiếp.")

## 2 — Google Drive  ← *đây là chỗ bạn cần bấm*

Kết quả lưu lên Drive nên mất session không mất gì. Chạy lại Run all là tiếp tục
từ chỗ dở.

Popup hiện ra → **Connect to Google Drive** → chọn tài khoản → kéo xuống → **Allow**.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive/gtsrb")
(DRIVE / "artifacts" / "runs").mkdir(parents=True, exist_ok=True)

done = [d for d in sorted((DRIVE / "artifacts" / "runs").glob("*"))
        if (d / "result.json").exists()]
print(f"Thư mục Drive : {DRIVE}")
print(f"Run đã có     : {len(done)}")
if done:
    print("  (những run này sẽ được BỎ QUA, không chạy lại)")

## 3 — Tải code

Lấy thẳng từ GitHub nên không phải upload file gì. Chạy lại thì nó `git pull` để
lấy bản mới nhất.

In [ ]:
import os, shutil, subprocess
from pathlib import Path

REPO = "https://github.com/tuanphongnguyen084/gtsrb-model-comparison.git"
PROJECT = Path("/content/DLprj")

if (PROJECT / ".git").exists():
    print("Đã có code, cập nhật bản mới nhất...")
    subprocess.run(["git", "-C", str(PROJECT), "pull", "--quiet"], check=False)
else:
    if PROJECT.exists():
        shutil.rmtree(PROJECT)
    print("Tải code từ GitHub...")
    subprocess.run(["git", "clone", "--quiet", "--depth", "1", REPO, str(PROJECT)],
                   check=True)

os.chdir(PROJECT)

# artifacts/ trỏ sang Drive -> kết quả sống sót khi mất session
local = PROJECT / "artifacts"
if not local.is_symlink():
    if local.exists():
        shutil.rmtree(local)
    local.symlink_to(DRIVE / "artifacts")

# Kiểm symlink thật sự ghi được sang Drive
probe = local / ".probe"
probe.write_text("ok")
assert (DRIVE / "artifacts" / ".probe").read_text() == "ok", "symlink hỏng"
probe.unlink()

print("thư mục làm việc :", os.getcwd())
print("artifacts/       -> Drive  (đã kiểm ghi được)")

## 4 — Cài gói còn thiếu

Chỉ `thop` và `onnx`. Mọi thứ khác Colab có sẵn.

★ **Không `pip install torch`** — sẽ kéo về bản không khớp CUDA và bạn train trên CPU
mà không hề biết.

In [ ]:
!pip install -q -r requirements-colab.txt
import thop, torch
print("thop", thop.__version__, "| torch", torch.__version__,
      "| CUDA vẫn OK:", torch.cuda.is_available())

## 5 — Tải và tiền xử lý dữ liệu  (~5 phút)

Tải thẳng trên Colab (mạng ~100 MB/s) chứ không upload từ máy bạn.

Dữ liệu để ở `/content` chứ **không** để trên Drive: đọc từ Drive lúc train chậm hơn
nhiều vì Drive gắn qua mạng. Chỉ kết quả mới đáng để trên Drive.

In [ ]:
!python scripts/prepare_data.py --download --preprocess clahe --img-size 48

Ô dưới **chặn** notebook nếu dữ liệu không sạch — thà dừng còn hơn chạy 2 tiếng
rồi phát hiện số liệu vô giá trị.

In [ ]:
import sys; sys.path.insert(0, "src")
import pandas as pd
from gtsrb.data.split import check_leakage

frame = pd.read_csv("data/processed/index.csv")
report = check_leakage(frame)
counts = frame["split"].value_counts()

print(f"Tổng ảnh         : {len(frame):,}   (phải 51.839)")
print(f"train/val/test   : {counts['train']:,} / {counts['val']:,} / {counts['test']:,}")
print(f"Track dùng chung : {report['n_shared_tracks']}   (PHẢI là 0)")
print(f"Lệch phân phối   : {report['max_class_ratio_gap_pct']:.3f}%   (phải < 1%)")

assert len(frame) == 51_839, "Thiếu ảnh — xem docs/SU_CO.md mục 1"
assert report["n_shared_tracks"] == 0, "CÓ RÒ RỈ DỮ LIỆU — dừng lại"
print("\nDữ liệu sạch. Chạy tiếp.")

## 6 — ABLATION 30 run  (~1,5–2 giờ)

Đây là phần lâu nhất. Sáu trục: độ phân giải, augmentation, tiền xử lý, model scaling,
label smoothing, và từng thành phần kiến trúc của M2.

Dùng **ngân sách 15 epoch** — đủ để XẾP HẠNG các biến thể, tiết kiệm ~60% thời gian.

★ Chạy lại an toàn: run nào đã có `result.json` trên Drive sẽ bị bỏ qua.

In [ ]:
!python scripts/run_ablation.py --axes all --budget --set train.num_workers=2

## 7 — Thêm 2 seed cho M2  (~25 phút)

Để trả lời: *chênh lệch giữa các model có vượt nhiễu khởi tạo không?*
Hiện nhóm mới chạy một seed — đó là hạn chế phương pháp lớn nhất.

In [ ]:
!python scripts/run_seeds.py --config configs/m2_vggres.yaml --seeds 43 44 \
    --set train.num_workers=2

## 8 — Train nốt M3 nếu còn thiếu

M3 ResNet18, MobileNetV2 và EfficientNet-B0 đã train xong trên máy cá nhân rồi.
Ô này chỉ chạy nếu phát hiện thiếu — bình thường nó sẽ in *"đủ rồi"* và bỏ qua.

In [ ]:
import subprocess
from pathlib import Path

NEEDED = ["m3_resnet18", "m3_mobilenetv2", "m3_effnetb0"]
missing = [n for n in NEEDED
           if not any((d / "result.json").exists()
                      for d in Path("artifacts/runs").glob(f"{n}_s42_2*"))]

if not missing:
    print("Đủ cả 3 backbone M3 rồi — bỏ qua.")
else:
    print("Thiếu:", missing)
    for name in missing:
        print(f"\n>>> train {name}")
        subprocess.run(["python", "scripts/train.py",
                        "--config", f"configs/{name}.yaml",
                        "--set", "train.num_workers=2", "train.batch_size=96"])

## 9 — Đánh giá trên tập test

Bắt buộc — không có số test thì bảng ablation trống. Bỏ McNemar vì phần đó chạy
trên máy cá nhân khi tổng hợp.

In [ ]:
!python scripts/evaluate.py --runs "artifacts/runs/*" --no-mcnemar

## 10 — Tải kết quả về máy

Chỉ đóng gói `result.json`, `config.yaml`, `train_log.csv` — **không lấy checkpoint**.
Ablation chỉ cần số liệu, mà 30 checkpoint là hàng GB. Gói này chỉ vài trăm KB.

File sẽ tự tải xuống thư mục `Downloads` của bạn.

In [ ]:
import zipfile
from pathlib import Path

KEEP = ("result.json", "config.yaml", "train_log.csv")
out = Path("/content/ketqua_colab.zip")

runs = [r for r in sorted(Path("artifacts/runs").glob("*"))
        if (r / "result.json").exists()]

with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for run in runs:
        for name in KEEP:
            p = run / name
            if p.exists():
                z.write(p, f"artifacts/runs/{run.name}/{name}")

print(f"Đóng gói {len(runs)} run -> {out.name}  ({out.stat().st_size/1024:.0f} KB)")

# Thống kê nhanh cho biết đã có đủ chưa
import re
axes = ("resolution", "augmentation", "preprocess", "scaling",
        "label_smoothing", "components")
n_abl = sum(1 for r in runs if any(a in r.name for a in axes))
n_seed = sum(1 for r in runs if "seed4" in r.name)
print(f"  ablation : {n_abl}/30")
print(f"  seed     : {n_seed}")
print(f"  model chính: {sum(1 for r in runs if not any(a in r.name for a in axes) and 'seed4' not in r.name)}")

from google.colab import files
files.download(str(out))

---

# Xong — làm gì tiếp trên máy bạn

```bash
cd ~/Downloads/DLprj
python scripts/merge_runs.py ~/Downloads/ketqua_colab.zip

python scripts/run_ablation.py --collect-only   # bảng + 6 hình ablation
python scripts/run_seeds.py --collect-only      # mean±std + kết luận nhiễu seed
make report baocao                              # sinh lại 2 báo cáo
```

`merge_runs.py` chỉ chép run chưa có và kiểm `result.json` hợp lệ trước khi chép.

---

## Nếu có trục trặc

| Hiện tượng | Xử lý |
|---|---|
| Ô 1 dừng, báo chưa có GPU | `Runtime → Change runtime type → T4 GPU → Save`, rồi Run all lại |
| Mất kết nối giữa chừng | Bấm Run all lại. Kết quả nằm trên Drive, nó bỏ qua phần đã xong |
| Colab báo hết hạn GPU | Bản miễn phí giới hạn ~12h/ngày. Hôm sau Run all lại, kết quả vẫn còn |
| Tab bị đóng | Colab chạy thêm ~90 phút rồi ngắt. Mở lại tab để giữ phiên |

**Trong lúc chạy: đừng đóng tab, đừng để máy ngủ.**

## Vì sao không đo tốc độ ở đây

Latency của T4 không nói gì về thiết bị biên. `make speed` và `make edge` phải chạy
trên **máy đích**, và lúc đo phải tắt mọi thứ khác — nên hai phần đó giữ ở máy cá nhân.